# thui-b94-q35-122b-smoke (Thuitanium / Knowless Crew) — B81 with the served model swapped to Qwen3.5-122B-A10B-NVFP4

**Smoke: 3 games at 1800 s. Numbers are not a score.**

**This is a Knowless Crew / Thuitanium experiment notebook.** The B81 solver, prompts, clock and vLLM profile
(context 32,768 / KV 7 GiB / MTP 0 / max_num_seqs 28) stay fixed. The only experiment change is the served
model, supplied by the `ippeiogawa/qwen35-122b-a10b-nvfp4` dataset and patched into a writable serving-bundle overlay in cell 9.

Serving stack by [Keith Tyser](https://www.kaggle.com/code/keithtyser/duck-qwen3-8-flash-next-nvfp4-mtp), harness by
[Tufa Labs](https://www.kaggle.com/code/jeroencottaar/tufa-labs-duck-harness-june-30-milestone-winner), anim solver
bundle `jakobbrggen/taaf-kaggle-source-anim-20260807-anim`.


## Upstream notes — the Tufa Labs duck harness (their text, reworded in the third person)

The duck harness notebook this fork descends from is Tufa Labs' "duck harness" (their June 30 milestone
winner). Their own note on it: the readable notebook scored Tufa Labs' milestone-winning 1.21, and later
runs of it did not repeat that result; the original, less readable notebook is also shared at
https://www.kaggle.com/code/jeroencottaar/taaf-duck-harness-kaggle and is not recommended.

- Tufa Labs' writeup of what the solver does: https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3/discussion/717133
- Machine Learning Street Talk interview by Tim Scarfe about the duck harness: https://x.com/MLStreetTalk/status/2072326433922297975?s=20

The solver was written by the Tufa Labs team; in alphabetical order: Harold Bessis, Jeroen Cottaar,
Isaiah Pressman, Andries Smit, Michal Tesnar, and Stefano Viel. The notebook holds infrastructure and
diagnostics only; the solver code lives in the attached source bundle. It installs the ARC runtime from the
competition wheelhouse, makes the bundled source snapshot importable, runs the solver setup commands, loads
the pickled benchmark, plays the competition games, and writes results to `/kaggle/working`. Diagnostics are
minimised during a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`) and kept full otherwise. A copy of
this notebook must select the RTX Pro 6000 GPU manually.


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
# thui-animfast: full diagnostics on an interactive public run (usage/events/transcript sidecars); minimal in a rerun.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv7-bf16-mtp0-c28-cg32'   # thui-a5: was kv5-bf16-mtp3-c8-cg32
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "7516192768",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "28",
    "TAAF_VLLM_MTP_TOKENS": "0",
    "TAAF_VLLM_OMP_THREADS": "1"
}
assert PUBLIC25_VLLM_PROFILE_ENV["TAAF_VLLM_KV_CACHE_MEMORY_BYTES"] == str(7 * 1024 ** 3)
print("THUI_A5_PROFILE ok mtp=0 kv=7GiB seqs=28", flush=True)
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# thui-animfast: resolve the competition mount instead of assuming its layout -- Kaggle serves either
# /kaggle/input/competitions/<comp> or /kaggle/input/<comp>, and which one varies between runs.
_COMP_CANDIDATES = ["/kaggle/input/competitions/arc-prize-2026-arc-agi-3", "/kaggle/input/arc-prize-2026-arc-agi-3"]
_COMP_DIR = next((_p for _p in _COMP_CANDIDATES if os.path.isdir(_p)), None)
assert _COMP_DIR is not None, (
    "thui-animfast: no competition mount found. Tried " + repr(_COMP_CANDIDATES)
    + "; /kaggle/input holds "
    + repr(sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else "MISSING")
)
_WHEELS = os.path.join(_COMP_DIR, "arc_agi_3_wheels")
assert os.path.isdir(_WHEELS), "thui-animfast: resolved wheels dir is not a directory: " + _WHEELS
print("thui-animfast: competition mount = " + _COMP_DIR, flush=True)
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        _WHEELS,
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1", "jakobbrggen/taaf-kaggle-source-anim-20260807-anim", "ippeiogawa/qwen35-122b-a10b-nvfp4"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir(label: str) -> Path:
    # thui-animfast: TWO attached datasets carry the marker (his June duck bundle and the anim bundle), so
    # "first marker wins" is a coin flip -- pick by the benchmark_label the marker file records.
    found = {}
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        try:
            found[json.loads(marker.read_text())["benchmark_label"]] = marker.parent
        except Exception as exc:
            print(f"thui-animfast: unreadable marker {marker}: {exc!r}", flush=True)
    if label not in found:
        raise RuntimeError(f"TAAF source bundle {label!r} not found under /kaggle/input; markers = {found}")
    return found[label]


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir("duck-harness-kaggle")          # his: serving_setup.py, vllm patches, watchdog, teardown
ANIM_BUNDLE_DIR = _find_bundle_dir("anim-20260807-anim")     # ours: the solver tree + its pickled benchmark / target
assert BUNDLE_DIR != ANIM_BUNDLE_DIR, "thui-animfast: both labels resolved to one directory"
assert (BUNDLE_DIR / "serving_setup.py").is_file(), f"thui-animfast: his bundle has no serving_setup.py: {BUNDLE_DIR}"
assert (ANIM_BUNDLE_DIR / "src" / "ARC3-Inference" / "inference" / "utils" / "animation.py").is_file(), (
    f"thui-animfast: the anim bundle has no animation.py: {ANIM_BUNDLE_DIR}")
print(f"thui-animfast: anim bundle = {ANIM_BUNDLE_DIR}", flush=True)
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands â€” installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
# thui-animfast: his tree minus the two solver repos (the June duck), plus the anim solver tree. The loop below
# inserts each entry at sys.path[0], so the LAST entries win -- the anim ones; the .pth is written anim-first.
_SOLVER_REPOS = {"ARC3-Inference", "tufa-arc-agi-framework"}
_his_entries = [e for e in _source_path_entries(BUNDLE_DIR) if e.parent.name not in _SOLVER_REPOS and e.name not in _SOLVER_REPOS]
_anim_entries = _source_path_entries(ANIM_BUNDLE_DIR)
assert _anim_entries and all(str(e).startswith(str(ANIM_BUNDLE_DIR)) for e in _anim_entries), _anim_entries
assert not any(("ARC3-Inference" in str(e) or "tufa-arc-agi-framework" in str(e)) for e in _his_entries), _his_entries
source_entries = _his_entries + _anim_entries
print(f"thui-animfast: source roots his={[str(e) for e in _his_entries]} anim={[str(e) for e in _anim_entries]}", flush=True)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in (_anim_entries + _his_entries)))   # anim first for child processes
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# ---- thui-b94: Qwen3.5 candidate serving overlay.
import hashlib as _hashlib
import shutil as _shutil
_b94_src = BUNDLE_DIR
_b94_dir = WORKING_DIR / "thui-b94-bundle-overlay"
if _b94_dir.exists():
    _shutil.rmtree(_b94_dir)
_b94_dir.mkdir(parents=True)
for _entry in _b94_src.iterdir():
    if _entry.name not in ("serving_setup.py", "SOURCE_IDENTITY.json"):
        (_b94_dir / _entry.name).symlink_to(_entry)
def _replace_once(src: str, old: str, new: str, label: str) -> str:
    count = src.count(old)
    assert count == 1, f"{label}: expected one anchor, found {count}"
    return src.replace(old, new)

def _replace_section(src: str, start: str, end: str, replacement: str, label: str) -> str:
    assert src.count(start) == 1, f"{label}: start anchor count != 1"
    assert src.count(end) == 1, f"{label}: end anchor count != 1"
    before, tail = src.split(start, 1)
    _discarded, after = tail.split(end, 1)
    return before + replacement + end + after

MODEL_CONSTANTS = 'MODEL_HF_REPO = "ippeiogawa/qwen35-122b-a10b-nvfp4"\nMODEL_DATASET = "ippeiogawa/qwen35-122b-a10b-nvfp4"\nMODEL_CONFIG_SHA256 = "4bbd5a6e8662d412cb6e57efeee5dfd4f8e13dc07e2b43398994f85a58f9993d"\nMODEL_QUANT_CONFIG_SHA256 = "0ee08c97b6123d808734e28951385ce0722c61737ac8c14a9f9a9e3889646c83"\nMODEL_WEIGHT_SHARD_COUNT = 9\nMODEL_WEIGHT_BYTES = 83_495_005_008\n\n'
SOURCE_IDENTITY = 'def source_identity() -> dict[str, Any]:\n    path = BUNDLE_DIR / "SOURCE_IDENTITY.json"\n    value = read_json(path)\n    if not isinstance(value, dict):\n        raise RuntimeError(f"Invalid source identity: {path}")\n    expected_setup = str(value.get("serving_setup_sha256", ""))\n    actual_setup = sha256_file(Path(__file__).resolve())\n    if not expected_setup or expected_setup != actual_setup:\n        raise RuntimeError(\n            f"Serving setup identity mismatch: {actual_setup} != {expected_setup or \'<missing>\'}"\n        )\n    if value.get("runtime_manifest_sha256") != VLLM_RUNTIME_MANIFEST_SHA256:\n        raise RuntimeError("Source identity has the wrong vLLM runtime manifest hash.")\n    runtime = value.get("runtime") or {}\n    if runtime.get("image") != VLLM_IMAGE or runtime.get("kaggle_dataset") != RUNTIME_DATASET:\n        raise RuntimeError("Source identity has the wrong vLLM runtime.")\n    return value\n\n\n'
MODEL_FUNCTIONS = 'def resolve_model_dir() -> Path:\n    mapped = input_paths().get(MODEL_DATASET)\n    candidates = [\n        mapped,\n        Path("/kaggle/input/qwen35-122b-a10b-nvfp4"),\n        Path("/kaggle/input/datasets/ippeiogawa/qwen35-122b-a10b-nvfp4"),\n    ]\n    matches = []\n    for candidate in candidates:\n        if candidate is None:\n            continue\n        if (candidate / "config.json").is_file() and (candidate / "hf_quant_config.json").is_file():\n            resolved = candidate.resolve()\n            if resolved not in matches:\n                matches.append(resolved)\n    if len(matches) != 1:\n        raise FileNotFoundError(f"Could not resolve one pinned {MODEL_DATASET}; matches={matches}")\n    return matches[0]\n\n\ndef verify_model(model_dir: Path, *, full_file_hashes: bool = True) -> dict[str, Any]:\n    del full_file_hashes\n    config_path = model_dir / "config.json"\n    quant_path = model_dir / "hf_quant_config.json"\n    if sha256_file(config_path) != MODEL_CONFIG_SHA256:\n        raise RuntimeError("Pinned model config hash does not match.")\n    if sha256_file(quant_path) != MODEL_QUANT_CONFIG_SHA256:\n        raise RuntimeError("Pinned ModelOpt quantization config hash does not match.")\n    config = read_json(config_path)\n    quant = read_json(quant_path)\n    text_config = config.get("text_config") or {}\n    quantization = config.get("quantization_config") or {}\n    quant_details = quant.get("quantization") or {}\n    if config.get("architectures") != ["Qwen3_5MoeForConditionalGeneration"]:\n        raise RuntimeError(f"Wrong model architecture: {config.get(\'architectures\')}")\n    if config.get("model_type") != "qwen3_5_moe" or text_config.get("model_type") != "qwen3_5_moe_text":\n        raise RuntimeError("Wrong Qwen3.5 model type identity.")\n    if quantization.get("quant_method") != "modelopt" or quantization.get("quant_algo") != "NVFP4":\n        raise RuntimeError(f"Wrong in-model quantization identity: {quantization}")\n    if quant_details.get("quant_algo") != "NVFP4" or int(quant_details.get("group_size", -1)) != 16:\n        raise RuntimeError(f"Wrong ModelOpt NVFP4 details: {quant_details}")\n    if int(text_config.get("mtp_num_hidden_layers", -1)) != 1:\n        raise RuntimeError("Wrong candidate MTP identity.")\n    required = ("model.safetensors.index.json", "chat_template.jinja")\n    for name in required:\n        if not (model_dir / name).is_file():\n            raise RuntimeError(f"Incomplete checkpoint; missing {name}")\n    shards = sorted(model_dir.glob("model-*-of-00009.safetensors"))\n    total = sum(path.stat().st_size for path in shards)\n    if len(shards) != MODEL_WEIGHT_SHARD_COUNT or total != MODEL_WEIGHT_BYTES:\n        raise RuntimeError(\n            f"Checkpoint shard identity mismatch: count={len(shards)} bytes={total}"\n        )\n    result = {\n        "dataset": MODEL_DATASET,\n        "config_sha256": MODEL_CONFIG_SHA256,\n        "quant_config_sha256": MODEL_QUANT_CONFIG_SHA256,\n        "weight_shard_count": len(shards),\n        "weight_bytes": total,\n        "architecture": config["architectures"][0],\n        "quant_algo": quantization["quant_algo"],\n        "mtp_num_hidden_layers": text_config["mtp_num_hidden_layers"],\n        "payload_check_deferred_to_vllm_load": True,\n    }\n    print("THUI_B94_SERVING ok model=Qwen3.5-122B-A10B-NVFP4", flush=True)\n    return result\n\n\n'

def patch(src: str) -> str:
    src = _replace_once(
        src,
        "Prepare the pinned Qwen3.8-Flash-Next vLLM server on Kaggle.",
        "Prepare the pinned Qwen3.5-122B-A10B-NVFP4 vLLM server on Kaggle.",
        "module description",
    )
    src = _replace_section(src, 'MODEL_HF_REPO = "RadixArk/', "\nVLLM_IMAGE =", MODEL_CONSTANTS, "model constants")
    src = _replace_once(src, 'SERVED_MODEL_NAME = "Qwen/Qwen3.8-Flash-Next-NVFP4"', 'SERVED_MODEL_NAME = "Qwen/Qwen3.5-122B-A10B-NVFP4"', "served model")
    src = _replace_once(src, 'DEFAULT_MTP_TOKENS = 3', 'DEFAULT_MTP_TOKENS = 0', "MTP default")
    src = _replace_section(src, "def source_identity()", "def resolve_runtime_dir()", SOURCE_IDENTITY, "source identity")
    src = _replace_section(src, "def resolve_model_dir()", "def _apply_whiteouts(", MODEL_FUNCTIONS, "model functions")
    # runtime_environment: targeted edits only -- drop the PLE source requirement, its hash check and the
    # PLE/RadixArk env; keep cache paths, allocator pin, cutlass checks and the vLLM version check unchanged.
    src = _replace_once(
        src,
        '    ple_path = (\n        site\n        / "vllm"\n        / "models"\n        / "qwen3_8_flash_next"\n'
        '        / "nvidia"\n        / "ple_layer.py"\n    )\n',
        "",
        "PLE source path",
    )
    src = _replace_once(src, '        site / "flashinfer" / "__init__.py",\n        ple_path,\n', '        site / "flashinfer" / "__init__.py",\n', "PLE required file")
    src = _replace_once(
        src,
        '    if sha256_file(ple_path) != VLLM_PLE_PATCHED_SHA256:\n'
        '        raise RuntimeError("The patched PLE source hash changed before import.")\n',
        "",
        "PLE source hash",
    )
    src = _replace_once(
        src,
        '            "VLLM_PLE_CPU_OFFLOAD": "1",\n'
        '            "VLLM_PLE_OFFLOAD_READY_TIMEOUT": str(SERVER_READY_TIMEOUT),\n',
        "",
        "PLE offload env",
    )
    src = _replace_once(
        src,
        '            "VLLM_RADIXARK_QWEN38_NVFP4_PLE_FP8": "1",\n'
        '            "VLLM_RADIXARK_QWEN38_NVFP4_CONFIG_SHA256": MODEL_CONFIG_SHA256,\n',
        "",
        "RadixArk runtime env",
    )
    src = _replace_once(src, '            "ple_patch_sha256": VLLM_PLE_PATCHED_SHA256,\n        }\n', "        }\n", "fast-mode PLE hash field")
    # The full deep-preload branch imports the Flash-Next PLE module; B94 runs fast start only.
    src = _replace_once(
        src,
        ") -> tuple[dict[str, str], dict[str, Any]]:\n",
        ") -> tuple[dict[str, str], dict[str, Any]]:\n"
        "    if deep_preload_validation:\n"
        '        raise RuntimeError("thui-b94: deep preload validation imports the Flash-Next PLE module; '
        'run with TAAF_KAGGLE_FAST_START=1")\n',
        "fast-start guard",
    )
    src = _replace_once(src, '        "ple_patch_sha256": sha256_file(ple_path),\n', "", "full-mode PLE hash field (unreachable under the fast-start guard)")
    # gpu_inventory: drop only the host-memory gate that exists for PLE CPU offload.
    src = _replace_once(
        src,
        '    if available < MIN_HOST_AVAILABLE_BYTES:\n'
        '        raise RuntimeError(\n'
        '            f"Host memory is too small for FP8 PLE CPU offload: {available} < {MIN_HOST_AVAILABLE_BYTES}"\n'
        '        )\n',
        "",
        "PLE host-memory gate",
    )
    src = _replace_once(src, '        "--distributed-executor-backend",\n        "mp",', '        "--distributed-executor-backend",\n        "mp",\n        "--trust-remote-code",', "trust remote code")
    src = _replace_once(src, "    ple_patch = patch_ple_layer()", '    ple_patch = {"skipped": True, "reason": "Qwen3.5 has no Flash-Next PLE gate"}', "PLE patch bypass")
    src = _replace_once(src, '        "model_hf_revision": MODEL_HF_REVISION,', '        "model_dataset": MODEL_DATASET,', "provenance model")
    src = _replace_once(src, '        "VLLM_RADIXARK_QWEN38_NVFP4_PLE_FP8": "1",\n        "VLLM_RADIXARK_QWEN38_NVFP4_CONFIG_SHA256": MODEL_CONFIG_SHA256,\n', '', "persisted RadixArk gates")
    src = _replace_section(
        src,
        '    log_text = SERVER_LOG.read_text(encoding="utf-8", errors="replace")\n    ple_log_patterns = {',
        '    value = {\n',
        '',
        "PLE readiness logs",
    )
    src = _replace_once(src, '        "ple_offload_log_matches": ple_log_matches,\n', '', "PLE capture result")
    return src

_b94_text = (_b94_src / "serving_setup.py").read_text()
_b94_text = patch(_b94_text)
(_b94_dir / "serving_setup.py").write_text(_b94_text)
_b94_ident = json.loads((_b94_src / "SOURCE_IDENTITY.json").read_text())
assert _b94_ident["serving_setup_sha256"] == _hashlib.sha256((_b94_src / "serving_setup.py").read_bytes()).hexdigest(), \
    "thui-b94: original serving identity mismatch"
_b94_ident["serving_setup_sha256"] = _hashlib.sha256((_b94_dir / "serving_setup.py").read_bytes()).hexdigest()
(_b94_dir / "SOURCE_IDENTITY.json").write_text(json.dumps(_b94_ident, indent=2, sort_keys=True) + "\n")
BUNDLE_DIR = _b94_dir
print(f"THUI_B94_OVERLAY ok bundle={BUNDLE_DIR} src={_b94_src}", flush=True)

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)
# ---- thui-animfast: the thui-v3 knobs, set AFTER his serving_setup persisted the analyzer env and BEFORE any
# `inference` import (tool_agent reads LOCAL_ANALYZER_SEED / YIELD_SECONDS at import time), then the graft teeth.
_KNOBS = {"LOCAL_ANALYZER_SEED": "20260825", "LOCAL_ANALYZER_YIELD_SECONDS": "180"}
_persisted = json.loads(SETUP_ENV_PATH.read_text())
assert _persisted.get("LOCAL_ANALYZER_MODEL_ID") == "Qwen/Qwen3.5-122B-A10B-NVFP4", _persisted.get("LOCAL_ANALYZER_MODEL_ID")
assert _persisted.get("LOCAL_ANALYZER_YIELD_SECONDS") == "60", "his serving_setup no longer persists yield 60 -- re-derive the override"
assert _persisted.get("LOCAL_ANALYZER_TEMPERATURE") == "0.6" and _persisted.get("MULTIMODAL_UPSCALE") == "4", _persisted
_persisted.update(_KNOBS)
SETUP_ENV_PATH.write_text(json.dumps(_persisted, indent=2, sort_keys=True) + "\n")
os.environ.update(_KNOBS)
assert "inference" not in sys.modules and "taaf" not in sys.modules, "solver imported before the knob override"
import inference.agent.tool_agent as _tool_agent
import inference.utils.animation as _anim_mod
import taaf as _taaf
for _m in (_tool_agent, _anim_mod, _taaf):
    assert str(Path(_m.__file__).resolve()).startswith(str(ANIM_BUNDLE_DIR.resolve())), (_m.__name__, _m.__file__)
assert _tool_agent._LOCAL_ANALYZER_SEED == int("20260825"), _tool_agent._LOCAL_ANALYZER_SEED
assert float(_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS) == float("180"), _tool_agent._LOCAL_ANALYZER_YIELD_SECONDS
assert os.environ["LOCAL_ANALYZER_MODEL_ID"] == "Qwen/Qwen3.5-122B-A10B-NVFP4"
print(f"THUI_ANIMFAST_GRAFT ok solver={Path(_tool_agent.__file__).parent} seed={_tool_agent._LOCAL_ANALYZER_SEED} "
      f"yield={_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS} model={os.environ['LOCAL_ANALYZER_MODEL_ID']} "
      f"temperature={os.environ['LOCAL_ANALYZER_TEMPERATURE']} upscale={os.environ['MULTIMODAL_UPSCALE']}", flush=True)


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(ANIM_BUNDLE_DIR / "deploy_target.pkl", "rb") as file:   # thui-animfast: the anim bundle's target (32400 s)
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(ANIM_BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:   # thui-animfast: the anim solver
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR
# thui-animfast: the unpickled solver must be the anim chassis, not the June duck.
assert bm.label == "anim-20260807-anim", bm.label
assert getattr(bm.solver, "animation_awareness", None) is True and getattr(bm.solver, "hard_noop_guard", None) is True, vars(bm.solver)
assert type(bm.solver).__module__ == "inference.framework.solver"
print(f"thui-animfast: bm.label={bm.label} solver={type(bm.solver).__name__} animation_awareness={bm.solver.animation_awareness} "
      f"hard_noop_guard={bm.solver.hard_noop_guard} target.max_runtime_s={target.max_runtime_s}", flush=True)


## 6. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts â€” the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# Exact public-25 and competition settings.
bm.solver.max_runtime_s_per_game = 7920.0
bm.solver.analyzer_timeout = 900.0
bm.solver.concurrency = 28
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = False
if float(getattr(target, 'max_runtime_s', 0.0) or 0.0) != 32400.0:
    raise RuntimeError(
        f'Expected the 32400-second notebook budget, got {target.max_runtime_s!r}.'
    )
print(
    f'PUBLIC25_SETTINGS budget_s={bm.solver.max_runtime_s_per_game} '
    f'concurrency={bm.solver.concurrency} analyzer_timeout={bm.solver.analyzer_timeout} '
    f'action_cap={bm.solver.max_actions_per_game} request_logs={bm.solver.save_request_logs}',
    flush=True,
)


## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise â€” an interactive "Save & Run" â€” play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Skip pre-run display and git-status copies; the staged identity pins the run.

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

PUBLIC_GAME_IDS = tuple(['tn36-ef4dde99', 'vc33-5430563c', 'bp35-0a0ad940'])   # thui-b94 smoke subset

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path(_COMP_DIR) / "environment_files")   # thui-animfast: resolved in cell 5
    offline_games = _offline_games(competition_env_files)
    offline_by_id = {game.env_name: game for game in offline_games}
    if len(offline_by_id) != len(offline_games):
        raise RuntimeError('The offline public game list contains duplicate IDs.')
    missing = sorted(set(PUBLIC_GAME_IDS) - set(offline_by_id))
    extra = sorted(set(offline_by_id) - set(PUBLIC_GAME_IDS))
    if missing or (extra and len(PUBLIC_GAME_IDS) == 25):
        raise RuntimeError(
            f'Offline public game set changed; missing={missing}, extra={extra}.'
        )
    bm.games = [offline_by_id[game_id] for game_id in PUBLIC_GAME_IDS]
    bm.solver.max_runtime_s_per_game = 1800.0   # thui-b94 smoke clock
    print(f"thui-b94: smoke {len(bm.games)} games @ {bm.solver.max_runtime_s_per_game} s", flush=True)
    if len(bm.games) != len(PUBLIC_GAME_IDS):
        raise RuntimeError(f'Expected 25 public games, got {len(bm.games)}.')
    print(f'PUBLIC25_SELECTION games={len(bm.games)} passes=1', flush=True)

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
if budget <= 600.0:
    raise RuntimeError(f'Notebook budget is too small for the teardown reserve: {budget}.')
soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(
    seconds=budget - 600.0
)

# Start recovery only after setup readiness and all run gates pass.
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
import vllm_server_watchdog as vllm_watchdog

vllm_watchdog_setup = vllm_watchdog.load_setup(BUNDLE_DIR / 'serving_setup.py')
vllm_watchdog.start_background(
    vllm_watchdog_setup,
    vllm_watchdog.WatchdogConfig(
        interval_seconds=15.0,
        request_timeout_seconds=5,
        failure_threshold=4,
        max_restart_attempts=2,
    ),
)

# Play the benchmark; watchdog stop and teardown run even if it raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=True)
    bm._save_json()
    if not TRUE_SUBMISSION:
        # Kaggle Save & Run expects this valid placeholder after an offline run.
        # A real competition rerun uses the live gateway and never enters this branch.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

        # Check terminal coverage, then call the frozen scorer once.
        # This path does not render HTML.
        public_runs = list(bm.game_runs)
        public_run_ids = [run.game_id for run in public_runs]
        if len(public_runs) != len(PUBLIC_GAME_IDS) or public_run_ids != list(PUBLIC_GAME_IDS):
            raise RuntimeError(
                f'Public run coverage changed: count={len(public_runs)} ids={public_run_ids}.'
            )
        unfinished = [
            (run.game_id, run.state, run.final_score)
            for run in public_runs
            if run.state not in {'won', 'gave_up', 'cancelled'}
            or run.final_score is None
        ]
        if unfinished:
            raise RuntimeError(f'Public runs did not finalize cleanly: {unfinished}.')
        crashed = [run.game_id for run in public_runs if run.state == 'crashed']
        if crashed:
            raise RuntimeError(f'Public runs crashed: {crashed}.')
        total_actions = sum(len(run.history) for run in public_runs)
        if total_actions <= 0:
            raise RuntimeError('Public runs produced no actions.')

        from inference.tools.eval import evaluate_runs, save_score_file

        score_summary = evaluate_runs([WORKING_DIR])
        score_path = save_score_file(
            score_summary,
            run_dirs=[WORKING_DIR],
            output_path=WORKING_DIR / "score.json",
        )
        if Path(score_path) != WORKING_DIR / 'score.json' or not Path(score_path).is_file():
            raise RuntimeError(f'Frozen scorer did not write score.json: {score_path}.')
        print(
            f'PUBLIC25_AUDIT runs=25 actions={total_actions} score_path={score_path}',
            flush=True,
        )
finally:
    try:
        vllm_watchdog.stop_background(timeout_seconds=15.0)
    finally:
        for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
            print(f"taaf.kaggle: teardown command: {command}", flush=True)
            subprocess.run(
                command,
                shell=True,
                check=False,
                cwd=WORKING_DIR,
                env=_command_env(),
                timeout=30.0,
            )


## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
# Minimal diagnostics are enabled; skip post-run HTML rendering.
